In [3]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
plt.style.use("bmh")

class CAPM:
    def __init__(self, stocks, start_date, end_date):
        self.data = None
        self.stocks = stocks
        self.start_date = start_date
        self.end_date = end_date
        
    def download_data(self):
        # Create an empty DataFrame first
        data = pd.DataFrame()
        
        for stock in self.stocks:
            # Set auto_adjust=False to get the 'Adj Close' column
            ticker = yf.download(stock, self.start_date, self.end_date, auto_adjust=False)
            # Add the stock's adjusted close prices as a column to the DataFrame
            data[stock] = ticker['Adj Close']
        
        return data
        
    def initialize(self):
        self.data = self.download_data()
        
        # we use monthly returns instead of daily returns
        self.data = self.data.resample('M').last()
        
        # print confirmation
        print("Data initialized successfully")
        print(self.data.head())
        
        # Rename the columns in self.data instead of creating a new DataFrame
        self.data = self.data.rename(columns={
            self.stocks[0]: 's_adjclose',
            self.stocks[1]: 'm_adjclose'
        })
        
        # log monthly returns
        self.data[['s_returns', 'm_returns']] = np.log(self.data[['s_adjclose', 'm_adjclose']])/self.data[['s_adjclose', 'm_adjclose']].shift(1)
        self.data = self.data[1:]

    def calculate_beta(self):
        # covariance matrix: the diag. items are the variances
        # off diagonals are covariances
        # the matrix is symmetric: cov[0,1] = cov[1,0]
        covariance_matrix = np.cov(self.data["s_returns"], self.data["m_returns"])

if __name__ == '__main__':
    capm = CAPM(['IBM', '^GSPC'], '2010-01-01', '2017-01-01')
    capm.initialize()

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

Data initialized successfully
                  IBM        ^GSPC
Date                              
2010-01-31  68.628731  1073.869995
2010-02-28  71.622383  1104.489990
2010-03-31  72.236313  1169.430054
2010-04-30  72.658752  1186.689941
2010-05-31  70.913857  1089.410034
            s_adjclose   m_adjclose  s_returns  m_returns
Date                                                     
2010-02-28   71.622383  1104.489990   0.062239   0.006525
2010-03-31   72.236313  1169.430054   0.059757   0.006396
2010-04-30   72.658752  1186.689941   0.059330   0.006053
2010-05-31   70.913857  1089.410034   0.058650   0.005893
2010-06-30   69.906151  1030.709961   0.059892   0.006369
...                ...          ...        ...        ...
2016-08-31  104.143921  2170.949951   0.044507   0.003535
2016-09-30  104.124268  2168.270020   0.044607   0.003538
2016-10-31  100.741928  2126.149902   0.044299   0.003534
2016-11-30  107.297935  2198.810059   0.046412   0.003620
2016-12-31  109.791496  2238.